In [ ]:
import torch
import os
import numpy as np
import matplotlib.pyplot as plt

from dotenv import load_dotenv
load_dotenv()

from sae_tools.model.load_model import load_decoder_matrix
from sae_tools.geometric.norm import analyze_matrix, visualize_norms
from sae_tools.geometric.umap import run_umap, visualize_density
from sae_tools.geometric.similarity import (
    plot_crosscorrelation_heatmap,
    plot_topk_similarity_heatmap
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Load $W_{dec}$

In [ ]:


SAE_ROOT = os.getenv("SAE_ROOT")
SAE_PATH = os.path.join(SAE_ROOT, "adamkarvonen/qwen3-8b-saes/saes_Qwen_Qwen3-8B_batch_top_k/resid_post_layer_18/trainer_2/ae.pt")

W_dec = load_decoder_matrix(SAE_PATH)

## 2. Check Norm

In [ ]:
norms = analyze_matrix(W_dec)
visualize_norms(norms)

print(f"\n{'='*40}")
print("Next Steps:")
print(f"Variable 'W_dec' is ready. You can now slice it to inspect specific features.")
print(f"Example: specific_feature = W_dec[4744]  # Get vector for feature #4744")
print(f"{'='*40}")

In [ ]:
data = W_dec.to(device)

norms = data.norm(p=2, dim=1, keepdim=True)
X_norm = data / norms

# 3. compute Gram Matrix (60k x 60k)
# Warning: 60k x 60k float32 matrix occupies about 14GB memory.
# If your memory < 16GB, this step will OOM (Out of Memory).
try:
    gram_matrix = torch.mm(X_norm, X_norm.t())
    print("Gram Matrix calculated successfully on GPU.")
    
    # for example, compute the average Cosine similarity
    mean_cosine = torch.mean(torch.abs(gram_matrix))
    print(f"Mean Absolute Cosine: {mean_cosine.item()}")
    
except RuntimeError as e:
    print(e)

## 3. Dimensional Reduction

In [ ]:
selected_indices = [4744, 9249, 14202, 24592, 65419]

# subset_data, subset_ids, is_seed = get_knn_subset(X_norm, selected_indices, k=10000)
# data = subset_data
# selected_indices = is_seed

x, y, z, embedding_numpy = run_umap(X_norm)

In [ ]:
visualize_density(x, y, z, selected_indices)

## 4. Visualize Similarity

In [ ]:
plot_crosscorrelation_heatmap(X_norm, selected_indices)

In [ ]:
plot_topk_similarity_heatmap(X_norm, selected_indices, k=50)